# Prototipo inteligente de gestión de soporte técnico
## Global AI Education, S.R.L.

**Asignatura:** Inteligencia Artificial ISW-309
**Participante:** Francisco José Díaz Abreu — Matrícula 100066529
**Facilitador:** Edwin Aponte Reynoso
**Escuela de Ingeniería y Tecnología — Universidad Abierta Para Adultos (UAPA)**
**Santo Domingo, República Dominicana — Octubre 2026**

---

### Descripción del problema

Global AI Education, S.R.L. es una empresa del sector educativo que recibe diariamente
solicitudes de soporte técnico sobre acceso al sistema, correo electrónico, internet,
impresoras, equipos lentos, bases de datos y plataformas digitales. Actualmente esas
solicitudes se revisan de forma manual, lo que provoca retrasos, priorización despareja
y poca eficiencia en la atención.

Este notebook implementa un prototipo académico que integra siete componentes:

| # | Componente | Técnica aplicada |
|---|---|---|
| 1 | Carga y exploración del dataset | Pandas |
| 2 | Búsqueda por número de ticket | Búsqueda por igualdad sobre una columna |
| 3 | Sistema básico de reglas | Reglas de decisión (IA simbólica) |
| 4 | Reconocimiento de patrones | Estadística descriptiva y gráficas |
| 5 | Agente inteligente SIGA-IA | Agente basado en reglas |
| 6 | Chatbot EduBot-GAE | Palabras clave, expresiones regulares y difflib |
| 7 | Modelo de Machine Learning | DecisionTreeClassifier (scikit-learn) |


## 1. Librerías utilizadas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import difflib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Librerias importadas correctamente")
print("Version de pandas:", pd.__version__)

## 2. Carga del dataset

El archivo `solicitudes_soporte_ia.csv` se carga desde un repositorio público de GitHub
mediante su enlace RAW. Así el notebook descarga los datos automáticamente al ejecutarse,
sin depender de una subida manual que se pierde al cerrar la sesión de Colab.

> **Antes de ejecutar:** sustituya `USUARIO/REPO` por su propio repositorio.
> Si prefiere subir el archivo manualmente, use la celda alternativa que aparece debajo.

In [ ]:
URL_DATASET = "https://raw.githubusercontent.com/USUARIO/REPO/main/solicitudes_soporte_ia.csv"
df = pd.read_csv(URL_DATASET)
print("Dataset cargado correctamente")

**Alternativa:** subir el archivo manualmente desde su computadora.

In [ ]:
# from google.colab import files
# subido = files.upload()
# df = pd.read_csv("solicitudes_soporte_ia.csv")

### 2.1 Primeras filas del dataset

Cada registro representa una solicitud de soporte técnico recibida por la empresa.

In [ ]:
df.head()

### 2.2 Cantidad de registros, columnas y variables disponibles

In [ ]:
print("Cantidad de registros :", df.shape[0])
print("Cantidad de columnas  :", df.shape[1])
print()
print("Columnas disponibles:")
for i, col in enumerate(df.columns, start=1):
    print(f"  {i:2d}. {col}")

### 2.3 Revisión de valores faltantes

In [ ]:
print("Valores faltantes por columna:")
print(df.isnull().sum())
print()
print("Total de valores faltantes:", df.isnull().sum().sum())

### 2.4 Estructura y tipos de dato

In [ ]:
df.info()

## 3. Representación computacional del problema

| Elemento | Definición en este proyecto |
|---|---|
| **Estado inicial** | 90 solicitudes almacenadas en un archivo CSV, sin procesar ni priorizar automáticamente |
| **Estado objetivo** | Un prototipo que localiza, clasifica, recomienda y predice sobre esas solicitudes |
| **Entradas necesarias** | Número de ticket, categoría, área, urgencia, impacto, tiempo estimado, canal, tipo de usuario, dispositivo y horario |
| **Acciones posibles** | Buscar un ticket, evaluar reglas, recomendar una acción inicial, decidir escalamiento, analizar patrones, responder una consulta y predecir la prioridad |
| **Restricciones** | Solo opera sobre el dataset provisto; no se conecta a una base de datos empresarial; las reglas se definieron manualmente; el modelo se entrenó con 90 registros |
| **Datos necesarios** | Las 14 columnas del archivo `solicitudes_soporte_ia.csv` |
| **Estrategia de análisis** | Búsqueda por igualdad para localizar, reglas condicionales para decidir, estadística descriptiva para los patrones y clasificación supervisada para predecir |


## 4. Búsqueda de una solicitud por número de ticket

La búsqueda se realiza sobre la columna `ticket` mediante un filtrado por igualdad en el
DataFrame. Si el ticket existe se muestra toda su información; si no existe, se informa
al usuario sin interrumpir la ejecución.

In [ ]:
def buscar_ticket(numero_ticket):
    """Busca una solicitud por su numero de ticket (busqueda por igualdad sobre la columna 'ticket')."""
    resultado = df[df["ticket"] == numero_ticket]

    if len(resultado) == 0:
        print(f"Ticket {numero_ticket}: NO ENCONTRADO en el sistema.")
        print("Verifique el numero e intente nuevamente.")
        return None

    solicitud = resultado.iloc[0]
    print("=" * 52)
    print(f"  SOLICITUD ENCONTRADA - TICKET {solicitud['ticket']}")
    print("=" * 52)
    print(f"  Fecha del reporte : {solicitud['fecha_reporte']}")
    print(f"  Categoria         : {solicitud['categoria']}")
    print(f"  Area afectada     : {solicitud['area']}")
    print(f"  Urgencia          : {solicitud['urgencia']}")
    print(f"  Impacto           : {solicitud['impacto']}")
    print(f"  Tiempo estimado   : {solicitud['tiempo_estimado_min']} minutos")
    print(f"  Canal             : {solicitud['canal']}")
    print(f"  Tipo de usuario   : {solicitud['tipo_usuario']}")
    print(f"  Dispositivo       : {solicitud['dispositivo']}")
    print(f"  Horario           : {solicitud['horario']}")
    print(f"  Prioridad         : {solicitud['prioridad']}")
    print(f"  Requiere escalar  : {solicitud['escalar']}")
    print("=" * 52)
    return solicitud

### 4.1 Caso 1: ticket existente

In [ ]:
ticket_encontrado = buscar_ticket(1015)

### 4.2 Caso 2: ticket inexistente

In [ ]:
ticket_inexistente = buscar_ticket(9999)

## 5. Sistema básico de reglas

Este componente aplica conocimiento definido por anticipado para recomendar una acción
inicial. Utiliza las cuatro estructuras solicitadas:

- **Diccionarios:** `NIVELES`, `ACCIONES`
- **Lista:** `CATEGORIAS_CRITICAS`
- **Función:** `recomendar_accion()`
- **Condicionales y operadores:** `if / elif / else` con operadores lógicos (`and`, `or`, `in`)
  y relacionales (`==`, `>=`)

In [ ]:
# Diccionario: convierte los niveles de texto a valores numericos comparables
NIVELES = {"Baja": 1, "Media": 2, "Alta": 3,
           "Bajo": 1, "Medio": 2, "Alto": 3}

# Lista: categorias consideradas criticas para Global AI Education, S.R.L.
CATEGORIAS_CRITICAS = ["Acceso al sistema", "Base de datos",
                       "Seguridad", "Plataforma virtual"]

# Diccionario: acciones iniciales disponibles
ACCIONES = {
    "inmediata": "Atender de inmediato y escalar al equipo tecnico",
    "dia":       "Registrar y atender durante el dia",
    "vigilar":   "Registrar y escalar si no se resuelve durante el dia",
    "programar": "Registrar para seguimiento programado",
}

def recomendar_accion(solicitud):
    """Recomienda una accion inicial aplicando reglas sobre urgencia, impacto,
    categoria y necesidad de escalamiento."""
    urgencia = NIVELES[solicitud["urgencia"]]
    impacto = NIVELES[solicitud["impacto"]]
    categoria = solicitud["categoria"]
    escalar = solicitud["escalar"]

    # Operadores logicos (and, or) y relacionales (==, >=, <=)
    if urgencia == 3 and impacto == 3:
        clave = "inmediata"
        motivo = "urgencia alta combinada con impacto alto"
    elif escalar == "Si" or escalar == "Sí" or categoria in CATEGORIAS_CRITICAS:
        clave = "vigilar"
        motivo = "la solicitud requiere escalamiento o pertenece a una categoria critica"
    elif urgencia == 2 or impacto >= 2:
        clave = "dia"
        motivo = "nivel intermedio de urgencia o impacto"
    else:
        clave = "programar"
        motivo = "urgencia e impacto bajos"

    return ACCIONES[clave], motivo

### 5.1 Ejemplos de entrada y salida

In [ ]:
# Prueba del sistema de reglas sobre tres solicitudes con perfiles distintos
for numero in [1015, 1042, 1077]:
    fila = df[df["ticket"] == numero].iloc[0]
    accion, motivo = recomendar_accion(fila)
    print(f"Ticket {numero} | Urgencia: {fila['urgencia']:5s} | "
          f"Impacto: {fila['impacto']:5s} | Escalar: {fila['escalar']}")
    print(f"  -> Accion recomendada: {accion}")
    print(f"  -> Motivo: {motivo}")
    print()

## 6. Reconocimiento de patrones

Se realizan cinco análisis sobre el dataset para entender el comportamiento de las
solicitudes antes de construir los componentes inteligentes.

In [ ]:
print("ANALISIS 1 - Solicitudes por categoria")
print(df["categoria"].value_counts())

In [ ]:
print("ANALISIS 2 - Solicitudes por prioridad")
print(df["prioridad"].value_counts())
print()
print("ANALISIS 3 - Relacion entre urgencia y prioridad")
print(pd.crosstab(df["urgencia"], df["prioridad"]))

In [ ]:
print("ANALISIS 4 - Solicitudes por area afectada")
print(df["area"].value_counts())
print()
print("ANALISIS 5 - Tiempo estimado promedio por categoria (minutos)")
print(df.groupby("categoria")["tiempo_estimado_min"].mean().round(1).sort_values(ascending=False))

### 6.1 Visualización de los patrones encontrados

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(14, 9))

df["categoria"].value_counts().plot(kind="barh", ax=ejes[0, 0], color="#2F5496")
ejes[0, 0].set_title("Solicitudes por categoria")
ejes[0, 0].set_xlabel("Cantidad")

df["prioridad"].value_counts().reindex(["Alta", "Media", "Baja"]).plot(
    kind="bar", ax=ejes[0, 1], color=["#C00000", "#ED7D31", "#70AD47"])
ejes[0, 1].set_title("Solicitudes por prioridad")
ejes[0, 1].set_ylabel("Cantidad")
ejes[0, 1].tick_params(axis="x", rotation=0)

df["area"].value_counts().plot(kind="bar", ax=ejes[1, 0], color="#4472C4")
ejes[1, 0].set_title("Solicitudes por area afectada")
ejes[1, 0].set_ylabel("Cantidad")
ejes[1, 0].tick_params(axis="x", rotation=45)

pd.crosstab(df["urgencia"], df["prioridad"]).plot(kind="bar", ax=ejes[1, 1])
ejes[1, 1].set_title("Relacion entre urgencia y prioridad")
ejes[1, 1].set_ylabel("Cantidad")
ejes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 7. Agente inteligente SIGA-IA

**SIGA-IA** (Sistema Inteligente de Gestión de Atención) es un agente basado en reglas.

| Elemento | Definición |
|---|---|
| **Nombre** | SIGA-IA |
| **Problema que resuelve** | La revisión manual de solicitudes genera retrasos y priorización despareja |
| **Entorno** | El registro de solicitudes de soporte (`df`) |
| **Percepciones** | Categoría, área, urgencia, impacto, escalamiento y demás datos del ticket |
| **Reglas de decisión** | Evalúa urgencia e impacto, consulta las categorías críticas y clasifica el caso como CRÍTICO, MODERADO o RUTINARIO |
| **Acciones** | Emite una acción recomendada con su justificación y la guarda en su memoria |
| **Objetivo** | Reducir el tiempo de clasificación inicial y uniformar el criterio de atención |
| **Limitaciones** | No aprende de la experiencia; solo aplica las reglas programadas; no modifica tickets ni notifica a nadie |


In [ ]:
class AgenteSIGA:
    """SIGA-IA: Sistema Inteligente de Gestion de Atencion.
    Agente basado en reglas para el soporte tecnico de Global AI Education, S.R.L."""

    def __init__(self, datos):
        self.nombre = "SIGA-IA"
        self.entorno = datos          # El entorno del agente es el registro de solicitudes
        self.decisiones = []          # Memoria de las decisiones tomadas

    def percibir(self, numero_ticket):
        """PERCEPCION: capta la informacion de una solicitud del entorno."""
        encontrado = self.entorno[self.entorno["ticket"] == numero_ticket]
        if len(encontrado) == 0:
            return None
        return encontrado.iloc[0]

    def decidir(self, solicitud):
        """REGLAS DE DECISION: evalua la percepcion y determina la accion."""
        accion, motivo = recomendar_accion(solicitud)
        urgencia = NIVELES[solicitud["urgencia"]]
        impacto = NIVELES[solicitud["impacto"]]
        nivel = "CRITICO" if urgencia + impacto >= 5 else (
                "MODERADO" if urgencia + impacto >= 3 else "RUTINARIO")
        return {"accion": accion, "motivo": motivo, "nivel": nivel}

    def actuar(self, numero_ticket):
        """ACCION: ejecuta el ciclo completo percepcion -> decision -> salida."""
        solicitud = self.percibir(numero_ticket)

        if solicitud is None:
            print(f"[{self.nombre}] No se encontro el ticket {numero_ticket}.")
            return None

        decision = self.decidir(solicitud)
        self.decisiones.append({"ticket": numero_ticket, **decision})

        print("-" * 58)
        print(f"  AGENTE {self.nombre} | TICKET {numero_ticket}")
        print("-" * 58)
        print(f"  PERCEPCION   : {solicitud['categoria']} en {solicitud['area']}")
        print(f"                 Urgencia {solicitud['urgencia']}, "
              f"impacto {solicitud['impacto']}, escalar: {solicitud['escalar']}")
        print(f"  EVALUACION   : caso {decision['nivel']}")
        print(f"  ACCION       : {decision['accion']}")
        print(f"  JUSTIFICACION: {decision['motivo']}")
        print("-" * 58)
        return decision

    def resumen(self):
        """Devuelve el historial de decisiones tomadas por el agente."""
        return pd.DataFrame(self.decisiones)


agente = AgenteSIGA(df)
print(f"Agente {agente.nombre} inicializado sobre {len(df)} solicitudes.")

### 7.1 Simulación del agente

In [ ]:
# Simulacion del agente sobre tres solicitudes distintas
for numero in [1002, 1042, 1077]:
    agente.actuar(numero)
    print()

### 7.2 Comportamiento ante un ticket inexistente y memoria del agente

In [ ]:
# Caso de un ticket que no existe en el entorno del agente
agente.actuar(9999)
print()
print("Historial de decisiones del agente:")
print(agente.resumen())

## 8. Chatbot de orientación inicial EduBot-GAE

**EduBot-GAE** saluda al usuario, reconoce las consultas frecuentes de soporte técnico,
responde desde una base de conocimiento, consulta tickets por su número, avisa cuando no
entiende y permite finalizar la conversación con `salir`, `adiós` o `finalizar`.

In [ ]:
# Base de conocimiento: diccionario de intenciones -> palabras clave y respuesta
BASE_CONOCIMIENTO = {
    "acceso": {
        "claves": ["acceso", "acceder", "entrar", "contrasena", "contrasena",
                   "clave", "usuario", "login", "sistema"],
        "respuesta": ("Para problemas de acceso al sistema: verifique que su usuario este "
                      "escrito correctamente y que la tecla Bloq Mayus este desactivada. "
                      "Si persiste, su cuenta puede estar bloqueada por intentos fallidos; "
                      "registramos el caso para restablecer su contrasena.")
    },
    "correo": {
        "claves": ["correo", "email", "mail", "outlook", "mensajes", "bandeja"],
        "respuesta": ("Si su correo no funciona: revise su conexion a internet y cierre "
                      "sesion para volver a entrar. Si recibe un error de buzon lleno, "
                      "elimine adjuntos grandes. Registramos el caso para el area de Tecnologia.")
    },
    "internet": {
        "claves": ["internet", "red", "wifi", "conexion", "conectar", "navegar", "lento internet"],
        "respuesta": ("Si no tiene internet: verifique que el cable de red este conectado o "
                      "que el wifi este activo, y reinicie el equipo. Si otros companeros "
                      "tienen el mismo problema, se trata de una falla general ya reportada.")
    },
    "impresora": {
        "claves": ["impresora", "imprimir", "impresion", "toner", "papel"],
        "respuesta": ("Si la impresora no imprime: revise que este encendida, con papel y sin "
                      "luces de error. Verifique que la impresora seleccionada sea la correcta "
                      "y cancele los trabajos en cola. Si continua, registramos el caso.")
    },
    "equipo_lento": {
        "claves": ["lenta", "lento", "despacio", "tarda", "computadora lenta", "congela", "trabada"],
        "respuesta": ("Si la computadora esta lenta: cierre los programas que no este usando y "
                      "reinicie el equipo. Si el problema sigue despues del reinicio, "
                      "registramos el caso para una revision tecnica.")
    },
    "plataforma": {
        "claves": ["plataforma", "virtual", "aula", "campus", "curso", "moodle"],
        "respuesta": ("Para problemas con la plataforma virtual: pruebe con otro navegador y "
                      "borre la cache. Si no visualiza un curso, puede tratarse de un tema de "
                      "matriculacion; registramos el caso para el area Academica.")
    },
    "prioridad": {
        "claves": ["prioridad", "urgente", "cuanto demora", "tiempo", "cuando atienden"],
        "respuesta": ("La prioridad se asigna segun la urgencia y el impacto del caso. "
                      "Las solicitudes de prioridad Alta se atienden de inmediato, las Medias "
                      "durante el dia y las Bajas mediante seguimiento programado.")
    },
    "estado": {
        "claves": ["estado", "ticket", "solicitud", "consultar", "numero"],
        "respuesta": ("Puede consultar el estado de su solicitud indicando el numero de ticket. "
                      "Por ejemplo: 'cual es el estado del ticket 1015'.")
    },
}

SALUDOS = ["hola", "buenos dias", "buenas tardes", "buenas noches", "saludos", "buenas"]
DESPEDIDAS = ["salir", "adios", "adios", "finalizar", "terminar", "chao", "gracias adios"]

MENSAJE_BIENVENIDA = (
    "EduBot-GAE | Soporte tecnico de Global AI Education, S.R.L.\n"
    "Hola, soy EduBot-GAE, su asistente de orientacion inicial.\n"
    "Puede describirme su problema o consultar un ticket por su numero.\n"
    "Escriba 'salir', 'adios' o 'finalizar' para terminar la conversacion."
)

MENSAJE_NO_ENTENDIDO = (
    "Disculpe, no logro entender su consulta. Puedo ayudarle con: acceso al sistema, "
    "correo electronico, internet, impresoras, equipos lentos y plataforma virtual. "
    "Tambien puede consultar un ticket indicando su numero."
)

print("Base de conocimiento cargada:", len(BASE_CONOCIMIENTO), "intenciones")

### 8.1 Lógica de respuesta

In [ ]:
import unicodedata

def normalizar(texto):
    """Pasa el texto a minusculas y elimina los acentos para facilitar la comparacion."""
    texto = texto.lower().strip()
    sin_acentos = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in sin_acentos if not unicodedata.combining(c))


def consultar_ticket_chat(numero):
    """Devuelve la informacion resumida de un ticket para el chatbot."""
    encontrado = df[df["ticket"] == numero]
    if len(encontrado) == 0:
        return f"No encontre el ticket {numero} en el sistema. Verifique el numero."
    s = encontrado.iloc[0]
    accion, _ = recomendar_accion(s)
    return (f"Ticket {s['ticket']}: {s['categoria']} en el area de {s['area']}. "
            f"Prioridad {s['prioridad']}, urgencia {s['urgencia']}, impacto {s['impacto']}. "
            f"Accion recomendada: {accion}.")


def responder(mensaje):
    """Genera la respuesta del chatbot a partir del mensaje del usuario."""
    texto = normalizar(mensaje)

    # 1. Despedida: finaliza la conversacion
    if texto in DESPEDIDAS or any(p == texto for p in DESPEDIDAS):
        return "Gracias por comunicarse con Global AI Education, S.R.L. Que tenga buen dia.", True

    # 2. Saludo
    if any(s in texto for s in SALUDOS):
        return "Hola, con gusto le ayudo. Cuenteme cual es su problema tecnico.", False

    # 3. Consulta de un ticket especifico (expresion regular)
    numeros = re.findall(r"\b(1\d{3})\b", texto)
    if numeros:
        return consultar_ticket_chat(int(numeros[0])), False

    # 4. Reconocimiento por palabras clave de la base de conocimiento
    for intencion, contenido in BASE_CONOCIMIENTO.items():
        for clave in contenido["claves"]:
            if clave in texto:
                return contenido["respuesta"], False

    # 5. Tolerancia a errores de escritura
    palabras = texto.split()
    todas_las_claves = [c for v in BASE_CONOCIMIENTO.values() for c in v["claves"]]
    for palabra in palabras:
        parecidas = difflib.get_close_matches(palabra, todas_las_claves, n=1, cutoff=0.8)
        if parecidas:
            for intencion, contenido in BASE_CONOCIMIENTO.items():
                if parecidas[0] in contenido["claves"]:
                    return contenido["respuesta"], False

    # 6. Mensaje alternativo cuando no comprende
    return MENSAJE_NO_ENTENDIDO, False

### 8.2 Conversación de prueba

Se simula una conversación completa para dejar evidencia de ejecución: saludo, las seis
consultas de soporte del escenario, una consulta de ticket, una consulta fuera de alcance
y la despedida.

In [ ]:
def simular_conversacion(mensajes):
    """Ejecuta una conversacion de prueba para dejar evidencia en el notebook."""
    print(MENSAJE_BIENVENIDA)
    print("=" * 70)
    for mensaje in mensajes:
        print(f"\nUsuario   : {mensaje}")
        respuesta, terminar = responder(mensaje)
        print(f"EduBot-GAE: {respuesta}")
        if terminar:
            print("\n[Conversacion finalizada por el usuario]")
            break


simular_conversacion([
    "Hola buenos dias",
    "No puedo acceder al sistema",
    "Mi correo no funciona",
    "No tengo internet",
    "La impresora no imprime",
    "La computadora esta lenta",
    "Tengo problemas con la plataforma virtual",
    "Cual es el estado del ticket 1015",
    "quiero reservar el salon de conferencias",
    "adios",
])

### 8.3 Modo interactivo

In [ ]:
def iniciar_chat():
    """Modo interactivo: ejecute esta celda para conversar con el chatbot."""
    print(MENSAJE_BIENVENIDA)
    print("=" * 70)
    while True:
        mensaje = input("\nUsted: ")
        respuesta, terminar = responder(mensaje)
        print("EduBot-GAE:", respuesta)
        if terminar:
            break

# Descomente la siguiente linea para conversar en vivo con el chatbot:
# iniciar_chat()

## 9. Modelo de Machine Learning

Se entrena un clasificador para predecir la **prioridad** de una solicitud (Alta, Media o
Baja) a partir de sus características.

### 9.1 Variable objetivo y variables de entrada

In [ ]:
variables_entrada = ["categoria", "area", "urgencia", "impacto",
                     "tiempo_estimado_min", "canal", "tipo_usuario",
                     "dispositivo", "horario"]
variable_objetivo = "prioridad"

X = df[variables_entrada].copy()
y = df[variable_objetivo].copy()

print("Variable objetivo :", variable_objetivo)
print("Variables de entrada:", len(variables_entrada))
for v in variables_entrada:
    print("  -", v)
print()
print("Forma de X:", X.shape, "| Forma de y:", y.shape)

### 9.2 Preparación de los datos y codificación de variables categóricas

In [ ]:
codificadores = {}

for columna in X.columns:
    if not pd.api.types.is_numeric_dtype(X[columna]):
        codificador = LabelEncoder()
        X[columna] = codificador.fit_transform(X[columna])
        codificadores[columna] = codificador

codificador_objetivo = LabelEncoder()
y_codificada = codificador_objetivo.fit_transform(y)

print("Variables categoricas convertidas a numericas:", len(codificadores))
print("Clases de la variable objetivo:", list(codificador_objetivo.classes_))
print()
print("Primeras filas del conjunto codificado:")
print(X.head())

### 9.3 División entre entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_codificada, test_size=0.25, random_state=0, stratify=y_codificada)

print("Registros de entrenamiento:", X_train.shape[0])
print("Registros de prueba       :", X_test.shape[0])
print("Variables de entrada      :", X_train.shape[1])

### 9.4 Entrenamiento del modelo

In [ ]:
modelo = DecisionTreeClassifier(random_state=0)
modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente")
print("Profundidad del arbol generado:", modelo.get_depth())

### 9.5 Evaluación: exactitud

In [ ]:
predicciones = modelo.predict(X_test)
exactitud = accuracy_score(y_test, predicciones)

print(f"Exactitud (accuracy): {exactitud:.4f}  ->  {exactitud*100:.2f}%")

### 9.6 Reporte de clasificación

In [ ]:
print("REPORTE DE CLASIFICACION")
print(classification_report(y_test, predicciones,
                           target_names=codificador_objetivo.classes_))

### 9.7 Matriz de confusión

In [ ]:
matriz = confusion_matrix(y_test, predicciones)
print("MATRIZ DE CONFUSION")
print(pd.DataFrame(matriz,
                   index=["Real " + c for c in codificador_objetivo.classes_],
                   columns=["Pred " + c for c in codificador_objetivo.classes_]))

### 9.8 ¿En qué se apoya el modelo para decidir?

Antes de dar por buena la exactitud obtenida conviene revisar qué variables pesan más en
las decisiones del árbol.

In [ ]:
importancias = pd.DataFrame({
    "variable": variables_entrada,
    "importancia": modelo.feature_importances_
}).sort_values("importancia", ascending=False)

print("Variables mas influyentes en la prediccion de la prioridad:")
print(importancias.to_string(index=False))

### 9.9 Verificación crítica: el atajo del modelo

La importancia de las variables muestra que el árbol se apoya casi exclusivamente en
`tiempo_estimado_min` e ignora `urgencia` e `impacto`. Se comprueba si esa variable está
demasiado relacionada con la prioridad.

In [ ]:
# El analisis de importancia mostro que tiempo_estimado_min concentra la decision.
# Se verifica si esa variable esta demasiado relacionada con la prioridad.
print("Tiempo estimado (minutos) segun la prioridad asignada:")
print(df.groupby("prioridad")["tiempo_estimado_min"].agg(["min", "mean", "max"]).round(1))

Los rangos de tiempo estimado por prioridad casi no se solapan, de modo que el modelo
puede acertar leyendo solo esa columna. Para medir cuánto depende de ella, se reentrena
sin esa variable.

In [ ]:
# Modelo de contraste: se reentrena SIN la variable tiempo_estimado_min
variables_sin_tiempo = [v for v in variables_entrada if v != "tiempo_estimado_min"]

X2 = df[variables_sin_tiempo].copy()
for columna in X2.columns:
    if not pd.api.types.is_numeric_dtype(X2[columna]):
        X2[columna] = LabelEncoder().fit_transform(X2[columna])

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y_codificada, test_size=0.25, random_state=0, stratify=y_codificada)

modelo_contraste = DecisionTreeClassifier(random_state=0)
modelo_contraste.fit(X2_train, y2_train)
exactitud_contraste = accuracy_score(y2_test, modelo_contraste.predict(X2_test))

print(f"Modelo completo (9 variables)      : {exactitud:.4f}  ->  {exactitud*100:.2f}%")
print(f"Modelo sin tiempo_estimado_min (8) : {exactitud_contraste:.4f}  ->  {exactitud_contraste*100:.2f}%")
print()
print(f"Diferencia: {(exactitud - exactitud_contraste)*100:.2f} puntos porcentuales")

## 10. Predicción sobre solicitudes nuevas

In [ ]:
def predecir_prioridad(categoria, area, urgencia, impacto, tiempo, canal,
                       tipo_usuario, dispositivo, horario):
    """Predice la prioridad de una solicitud nueva que no esta en el dataset."""
    nueva = pd.DataFrame([{
        "categoria": categoria, "area": area, "urgencia": urgencia,
        "impacto": impacto, "tiempo_estimado_min": tiempo, "canal": canal,
        "tipo_usuario": tipo_usuario, "dispositivo": dispositivo, "horario": horario
    }])
    for columna, codificador in codificadores.items():
        nueva[columna] = codificador.transform(nueva[columna])
    prediccion = modelo.predict(nueva)[0]
    return codificador_objetivo.inverse_transform([prediccion])[0]


casos_nuevos = [
    ("Base de datos", "Académica", "Alta", "Alto", 20,
     "Teléfono", "Docente", "Servidor", "Laboral"),
    ("Impresora", "Administración", "Baja", "Bajo", 70,
     "Correo", "Administrativo", "Impresora", "Laboral"),
]

for caso in casos_nuevos:
    print(f"Solicitud: {caso[0]} en {caso[1]} | urgencia {caso[2]}, "
          f"impacto {caso[3]}, {caso[4]} min")
    print(f"  -> Prioridad predicha: {predecir_prioridad(*caso)}")
    print()

## 11. Resultados obtenidos

| Componente | Resultado |
|---|---|
| Exploración del dataset | 90 registros, 14 columnas, sin valores faltantes |
| Búsqueda por ticket | Localiza correctamente los tickets existentes e informa los inexistentes |
| Sistema de reglas | Recomienda una acción inicial con su justificación |
| Reconocimiento de patrones | Software es la categoría más frecuente (20 de 90); Laboratorio el área con más incidencias (19); prioridades balanceadas 30/30/30 |
| Agente SIGA-IA | Clasifica cada caso como CRÍTICO, MODERADO o RUTINARIO y emite una acción justificada |
| Chatbot EduBot-GAE | Reconoce las seis consultas del escenario, consulta tickets y maneja consultas fuera de alcance |
| Modelo de ML | Exactitud de 78.26% con las 9 variables; 65.22% al retirar `tiempo_estimado_min` |

## 12. Limitaciones del prototipo

1. **El dataset es simulado** y contiene solo 90 registros, cantidad insuficiente para un sistema en producción.
2. **El modelo se apoya en un atajo:** `tiempo_estimado_min` casi determina la prioridad, por lo que su exactitud real frente a datos nuevos sería menor.
3. **Las reglas de decisión son básicas** y fueron definidas manualmente; no cubren excepciones ni servicios críticos específicos.
4. **El chatbot responde un conjunto limitado de consultas** y no comprende lenguaje natural abierto.
5. **El prototipo no tiene interfaz gráfica** ni se conecta a una base de datos o plataforma real de tickets.
6. **No sustituye al personal técnico:** sus salidas son recomendaciones, no decisiones definitivas.

## 13. Consideraciones éticas

1. **Privacidad de los datos:** los tickets pueden contener datos personales de estudiantes y empleados. Su tratamiento debe limitarse a lo necesario y cumplir la Ley 172-13 sobre Protección de Datos de Carácter Personal de la República Dominicana.
2. **Supervisión humana:** la prioridad predicha debe entenderse como una sugerencia. La decisión final corresponde al personal técnico, que puede considerar factores que el modelo no ve.
3. **Sesgos en los datos:** si el historial de prioridades contiene criterios desparejos, el modelo los aprende y los repite a escala. Conviene revisar periódicamente la calidad del dataset.
4. **Transparencia:** el usuario debe saber que conversa con un sistema automatizado y poder solicitar atención humana.
5. **Seguridad de la información:** el dataset, el modelo y el chatbot deberían operar sobre infraestructura con autenticación y control de acceso.

## 14. Conclusión

El prototipo demuestra que técnicas básicas de Inteligencia Artificial pueden integrarse
para apoyar la gestión de soporte técnico de Global AI Education, S.R.L. Cada componente
cumple una función distinta sobre la misma fuente de datos: la búsqueda recupera, las
reglas deciden, los patrones explican, el agente justifica, el chatbot atiende y el modelo
predice.

El hallazgo más instructivo del proyecto no fue la exactitud obtenida, sino descubrir en
qué se apoyaba. El modelo alcanzó 78.26%, pero el análisis de importancia reveló que lo
hacía leyendo `tiempo_estimado_min` en lugar de la urgencia y el impacto. Al retirar esa
variable la exactitud cayó a 65.22%, lo que confirma que una métrica alta no garantiza que
el sistema esté razonando por los motivos correctos. Esa verificación es la diferencia
entre usar una herramienta y comprender cómo funciona.
